In [10]:
import pandas as pd
import json

with open('merged_data.json') as f:
    data = json.load(f)

food_counts = pd.Series([record['food_type'] for record in data]).value_counts()
df = pd.DataFrame({'food_type': food_counts.index, 'count': food_counts.values})
df.to_csv('cuisines.csv', index=False)

In [11]:
# Load the cuisine mapping
with open('cuisine_mapping.json') as f:
    cuisine_mapping = json.load(f)

# Create a reverse mapping: food_type -> cuisine group
reverse_cuisine_map = {}
for group, cuisines in cuisine_mapping.items():
    for cuisine in cuisines:
        reverse_cuisine_map[cuisine] = group

# Add a new column with the cuisine group
df['cuisine_group'] = df['food_type'].apply(lambda x: reverse_cuisine_map.get(x, 'Others') if isinstance(x, str) else 'Others')

print(df)

                 food_type  count    cuisine_group
0                 American    865         American
1                  Italian    850          Italian
2    Contemporary American    649         American
3               Steakhouse    328      BBQ / Grill
4                  Seafood    267          Seafood
..                     ...    ...              ...
109       Regional Mexican      1  Mexican / Latin
110               Sicilian      1          Italian
111               Austrian      1         European
112                 Syrian      1    Mediterranean
113                Burmese      1           Others

[114 rows x 3 columns]


In [12]:
cuisine_group_counts = df['cuisine_group'].value_counts().reset_index()
cuisine_group_counts.columns = ['cuisine_group', 'count']
cuisine_group_counts = cuisine_group_counts.sort_values('count', ascending=False)
print(cuisine_group_counts)

       cuisine_group  count
0             Others     26
1           European     18
2    Mexican / Latin     15
3              Asian     13
4      Mediterranean     11
5           American     10
6        BBQ / Grill      5
7            Seafood      5
8            Italian      4
9   Health / Dietary      4
10          Japanese      3


In [13]:
countries_counts = pd.Series([record['country'] for record in data]).value_counts()
df = pd.DataFrame({'country': countries_counts.index, 'count': countries_counts.values})
print(df)

  country  count
0      US   5000


In [14]:
cities_counts = pd.Series([record['city'] for record in data]).value_counts()
df = pd.DataFrame({'city': cities_counts.index, 'count': cities_counts.values})
print(df)

              city  count
0         New York    695
1          Houston    175
2        San Diego    162
3           Denver    139
4    San Francisco    127
..             ...    ...
911      St Joseph      1
912          Tubac      1
913     Marysville      1
914         Monroe      1
915      Peekskill      1

[916 rows x 2 columns]


In [15]:
price_counts = pd.Series([record['price_range'] for record in data]).value_counts()
df = pd.DataFrame({'price_range': price_counts.index, 'count': price_counts.values})
print(df)

     price_range  count
0  $30 and under   3125
1     $31 to $50   1567
2   $50 and over    308


In [16]:
payment_options = {
    option
    for record in data
    for option in record.get('payment_options', [])
    if isinstance(record.get('payment_options', None), list)
}
print(sorted(payment_options))

['AMEX', 'Carte Blanche', 'Cash Only', 'Diners Club', 'Discover', 'JCB', 'MasterCard', 'Pay with OpenTable', 'Visa']


In [17]:
# static reverse map for known payment option strings; skip others
reverse_payment_map = {
    'AMEX': 'AMEX',
    'Visa': 'Visa',
    'MasterCard': 'MasterCard',
    'Discover': 'Discover',
    'Diners Club': 'Discover',
    'Carte Blanche': 'Discover',
}

In [18]:
def map_cuisine_subcategory(subcategory):
    if isinstance(subcategory, str):
        return reverse_cuisine_map.get(subcategory, 'Others')
    return 'Others'

def normalize_cuisine(cuisine):
    return cuisine if isinstance(cuisine, str) else 'Others'

def map_payment_labels(payments):
    labels = []
    seen = set()
    for payment in payments or []:
        label = reverse_payment_map.get(payment)
        if label and label not in seen:
            labels.append(label)
            seen.add(label)
    return labels

search_items = []
for record in data:
    search_items.append({
        'objectID': record.get('objectID'),
        'name': record.get('name'),
        'stars_count': record.get('stars_count'),
        'reviews_count': record.get('reviews_count'),
        'price_range': record.get('price_range'),
        'cuisine': normalize_cuisine(record.get('food_type')),
        'cuisine_cat': map_cuisine_subcategory(record.get('food_type')),
        'city': record.get('city'),
        'image_url': record.get('image_url'),
        'mobile_reserve_url': record.get('mobile_reserve_url'),
        'reserve_url': record.get('reserve_url'),
        '_geoloc': record.get('_geoloc', {}),
        'payment_options': map_payment_labels(record.get('payment_options', [])),
    })

with open('search_items.json', 'w') as f:
    json.dump(search_items, f, indent=2)